# TypiClust (TPC_RP) Implementation on CIFAR-10
**ML Coursework – K23115695**

Based on: Hacohen et al. (2022) *"Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets"*

**Kaggle Setup:**
- Runtime → GPU T4 ×2 or P100
- Internet must be ON (to download CIFAR-10 on first run)
- Checkpoints saved to `/kaggle/working/` (persists across saves)

**Stages:**
1. SimCLR self-supervised representation learning (ResNet-18)
2. Feature extraction
3. TypiClust active learning + baselines
4. Algorithm modification & evaluation
5. Plots & statistical analysis


## Cell 1 – Imports & Configuration

In [41]:
import os, time, json, random, copy, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Reproducibility ──
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

# ── ALL files save directly to /kaggle/working/ so they appear in Output ──
SAVE_DIR = '/kaggle/working'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"All files save to → {SAVE_DIR}")

Device: cuda
GPU   : Tesla T4
All files save to → /kaggle/working


## Cell 2 – Download CIFAR-10

In [42]:
DATA_DIR = '/kaggle/working/data'

trainset_raw = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True)
testset_raw  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True)

CIFAR10_CLASSES = trainset_raw.classes
print(f"Train: {len(trainset_raw)} | Test: {len(testset_raw)}")
print(f"Classes: {CIFAR10_CLASSES}")


Train: 50000 | Test: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Cell 3 – SimCLR Augmentation

In [43]:
class SimCLRAugmentation:
    """Return two differently-augmented views of the same image."""
    def __init__(self, size=32):
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.4914, 0.4822, 0.4465],
                                 [0.2470, 0.2435, 0.2616]),
        ])
    def __call__(self, x):
        return self.t(x), self.t(x)


class PairDataset(Dataset):
    """Wraps a dataset so each __getitem__ returns (view1, view2, label)."""
    def __init__(self, base, augment):
        self.base = base
        self.aug  = augment
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        img, lab = self.base[i]
        v1, v2 = self.aug(img)
        return v1, v2, lab

print("SimCLR augmentation ready.")


SimCLR augmentation ready.


## Cell 4 – SimCLR Model (ResNet-18)

In [44]:
class SimCLR(nn.Module):
    """
    ResNet-18 backbone (modified for 32×32 CIFAR images)
    + 2-layer MLP projection head → 128-dim.
    Penultimate layer (512-dim) is the feature representation.
    """
    def __init__(self, proj_dim=128):
        super().__init__()
        base = resnet18(weights=None)
        base.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)   # no aggressive downsample
        base.maxpool = nn.Identity()                           # skip maxpool for 32×32
        self.encoder = nn.Sequential(*list(base.children())[:-1])  # → (B, 512, 1, 1)
        self.projector = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(inplace=True),
            nn.Linear(512, proj_dim),
        )

    def forward(self, x):
        h = self.encoder(x).flatten(1)          # (B, 512)
        z = self.projector(h)                    # (B, 128)
        return h, z

    def features(self, x):
        return self.encoder(x).flatten(1)

print("SimCLR architecture ready.")


SimCLR architecture ready.


## Cell 5 – NT-Xent Contrastive Loss

In [45]:
class NTXent(nn.Module):
    def __init__(self, tau=0.5):
        super().__init__()
        self.tau = tau
        self.ce  = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        B = z_i.size(0)
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        z   = torch.cat([z_i, z_j], 0)                        # (2B, D)
        sim = z @ z.T / self.tau                               # (2B, 2B)
        # mask out self-similarity
        sim.fill_diagonal_(-1e9)
        labels = torch.cat([torch.arange(B, 2*B),
                            torch.arange(0, B)]).to(z.device)
        return self.ce(sim, labels)

print("NT-Xent loss ready.")


NT-Xent loss ready.


## Cell 6 – Train SimCLR (500 epochs with warmup, checkpoints every 10)

**Fix:** Added 10-epoch linear warmup (critical for SimCLR convergence).  
**Expected:** Loss should drop from ~6.3 → ~2.0-2.5.  
**Time:** ~11 hours on T4. If session dies, re-run and it resumes.

In [46]:
# Use the 'latest' version this time
src = '/kaggle/input/datasets/mariam445/simclr-model/simclr_latest.pt'
dest = os.path.join(SAVE_DIR, 'simclr_latest.pt')

if os.path.exists(src):
    shutil.copy(src, dest)
    print(f"✅ Loaded the 'latest' checkpoint from {src}")
else:
    print("❌ That file wasn't found in the dataset either!")
    
def train_simclr(epochs=200, bs=512, lr=0.4, ckpt_every=10):
    pair_ds = PairDataset(trainset_raw, SimCLRAugmentation())
    loader  = DataLoader(pair_ds, batch_size=bs, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)

    model     = SimCLR().to(DEVICE)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = NTXent(tau=0.5)

    # resume from checkpoint if available
    ckpt_path = os.path.join(SAVE_DIR, 'simclr_latest.pt')
    start, losses = 0, []
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optim'])
        scheduler.load_state_dict(ck['sched'])
        start  = ck['epoch'] + 1
        losses = ck['losses']
        print(f"Resuming from epoch {start}")

    if start >= epochs:
        print(f"Already trained {epochs} epochs, skipping.")
        return model, losses

    model.train()
    for ep in range(start, epochs):
        t0, running = time.time(), 0.0
        for v1, v2, _ in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            _, z1 = model(v1)
            _, z2 = model(v2)
            loss = criterion(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item()
        scheduler.step()
        avg = running / len(loader)
        losses.append(avg)

        print(f"Epoch {ep+1:>3}/{epochs} | loss {avg:.4f} | "
              f"lr {scheduler.get_last_lr()[0]:.5f} | {time.time()-t0:.0f}s")

        if (ep+1) % ckpt_every == 0:
            torch.save({'epoch': ep, 'model': model.state_dict(),
                        'optim': optimizer.state_dict(),
                        'sched': scheduler.state_dict(),
                        'losses': losses}, ckpt_path)
            print(f"   Checkpoint saved (epoch {ep+1})")

    # ── SAVE FINAL MODEL separately so it always appears in Output ──
    final_path = os.path.join(SAVE_DIR, 'simclr_final_model.pt')
    torch.save({'epoch': epochs-1, 'model': model.state_dict(), 'losses': losses}, final_path)
    print(f"FINAL MODEL SAVED → {final_path}")
    print(f"SimCLR training done, final loss {losses[-1]:.4f}")
    return model, losses


simclr_model, simclr_losses = train_simclr()

✅ Loaded the 'latest' checkpoint from /kaggle/input/datasets/mariam445/simclr-model/simclr_latest.pt
Resuming from epoch 200
Already trained 200 epochs, skipping.


## Cell 7 – SimCLR Training Loss

In [47]:
plt.figure(figsize=(8, 3.5))
plt.plot(simclr_losses, lw=1.2, color='#1f77b4')
plt.xlabel('Epoch'); plt.ylabel('NT-Xent Loss')
plt.title('SimCLR Training Loss - CIFAR-10 (ResNet-18, 200 epochs)')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_simclr_loss.png'), dpi=300)
plt.show()
print("Saved: plot_simclr_loss.png")

Saved: plot_simclr_loss.png


## Cell 8 – Extract L2-normalised Features (512-dim)

In [48]:
def extract_features(model, dataset, bs=256):
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616]),
    ])
    class Wrap(Dataset):
        def __init__(self, ds, tf): self.ds, self.tf = ds, tf
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            img, lab = self.ds[i]
            return self.tf(img), lab

    loader = DataLoader(Wrap(dataset, eval_tf), batch_size=bs,
                        shuffle=False, num_workers=2, pin_memory=True)
    model.eval()
    feats, labs = [], []
    with torch.no_grad():
        for x, y in loader:
            h = model.features(x.to(DEVICE))
            h = F.normalize(h, dim=1)
            feats.append(h.cpu().numpy())
            labs.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labs)


feat_train_path = os.path.join(SAVE_DIR, 'feat_train.npy')
lab_train_path  = os.path.join(SAVE_DIR, 'lab_train.npy')
feat_test_path  = os.path.join(SAVE_DIR, 'feat_test.npy')
lab_test_path   = os.path.join(SAVE_DIR, 'lab_test.npy')

if os.path.exists(feat_train_path):
    print("Loading cached features...")
    feat_train = np.load(feat_train_path)
    lab_train  = np.load(lab_train_path)
    feat_test  = np.load(feat_test_path)
    lab_test   = np.load(lab_test_path)
else:
    print("Extracting features...")
    feat_train, lab_train = extract_features(simclr_model, trainset_raw)
    feat_test,  lab_test  = extract_features(simclr_model, testset_raw)
    np.save(feat_train_path, feat_train)
    np.save(lab_train_path,  lab_train)
    np.save(feat_test_path,  feat_test)
    np.save(lab_test_path,   lab_test)
    print("Features saved!")

print(f"Train features: {feat_train.shape}  |  Test features: {feat_test.shape}")

Extracting features...
Features saved!
Train features: (50000, 512)  |  Test features: (10000, 512)


## Cell 9 – TypiClust (TPC_RP) – Original Algorithm

Three steps:
1. **Representation:** SimCLR features (already extracted)
2. **Clustering:** K-Means for diversity (`n_clusters = budget`)
3. **Typicality:** Select densest point per cluster (`K=20` nearest-neighbour inverse distance)


In [49]:
class TypiClust:
    """Original TPC_RP algorithm (Hacohen et al., 2022)."""

    def __init__(self, features, labels, K=20, max_clusters=500):
        self.features     = features
        self.labels       = labels
        self.N            = len(features)
        self.K            = K
        self.max_clusters = max_clusters

        # pre-compute typicality for every point
        k = min(K + 1, self.N)
        nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
        nn.fit(features)
        dists, _ = nn.kneighbors(features)
        avg_d = dists[:, 1:].mean(axis=1)           # skip self
        self.typicality = 1.0 / (avg_d + 1e-10)
        print(f"TypiClust ready – typicality range "
              f"[{self.typicality.min():.3f}, {self.typicality.max():.3f}]")

    # ── initial pool (Algorithm 1) ──
    def select(self, budget, labeled=None):
        labeled_set = set(labeled) if labeled else set()
        n_clust = min((len(labeled_set) + budget) if labeled else budget,
                      self.max_clusters)

        km = (KMeans(n_clust, n_init=10, random_state=42)
              if n_clust <= 50
              else MiniBatchKMeans(n_clust, batch_size=1024, n_init=3, random_state=42))
        cl = km.fit_predict(self.features)

        # find uncovered clusters (or all, if no labeled yet)
        candidates = []
        for cid in range(n_clust):
            idxs = np.where(cl == cid)[0]
            if len(idxs) < 5:
                continue
            if labeled_set and any(i in labeled_set for i in idxs):
                continue
            # best typical point
            best = idxs[np.argmax(self.typicality[idxs])]
            if best not in labeled_set:
                candidates.append((best, len(idxs)))

        # sort by cluster size descending (largest uncovered first)
        candidates.sort(key=lambda x: x[1], reverse=True)
        picked = [c[0] for c in candidates[:budget]]

        # fill if not enough
        if len(picked) < budget:
            pool = list(set(range(self.N)) - labeled_set - set(picked))
            order = np.argsort(self.typicality[pool])[::-1]
            for i in order:
                if len(picked) >= budget: break
                picked.append(pool[i])

        return picked[:budget]


print("TypiClust class defined.")


TypiClust class defined.


## Cell 10 – Baseline Strategies

In [50]:
class RandomAL:
    def __init__(self, N): self.N = N
    def select(self, budget, labeled=None):
        pool = list(set(range(self.N)) - (set(labeled) if labeled else set()))
        return list(np.random.choice(pool, min(budget, len(pool)), replace=False))


class UncertaintyAL:
    """Lowest max-softmax probability (needs a trained classifier)."""
    def __init__(self, N): self.N = N
    def select(self, budget, labeled=None, model=None, features=None):
        if model is None:  # first round – fall back to random
            pool = list(set(range(self.N)) - (set(labeled) if labeled else set()))
            return list(np.random.choice(pool, min(budget, len(pool)), replace=False))
        pool = list(set(range(self.N)) - set(labeled))
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(features[pool], dtype=torch.float32).to(DEVICE))
            conf = F.softmax(logits, dim=1).max(1)[0].cpu().numpy()
        order = np.argsort(conf)
        return [pool[i] for i in order[:budget]]


class MarginAL:
    """Smallest gap between top-2 softmax probabilities."""
    def __init__(self, N): self.N = N
    def select(self, budget, labeled=None, model=None, features=None):
        if model is None:
            pool = list(set(range(self.N)) - (set(labeled) if labeled else set()))
            return list(np.random.choice(pool, min(budget, len(pool)), replace=False))
        pool = list(set(range(self.N)) - set(labeled))
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(features[pool], dtype=torch.float32).to(DEVICE))
            probs = F.softmax(logits, dim=1).sort(dim=1, descending=True)[0]
            margin = (probs[:, 0] - probs[:, 1]).cpu().numpy()
        order = np.argsort(margin)
        return [pool[i] for i in order[:budget]]


print("Baselines defined: Random, Uncertainty, Margin.")


Baselines defined: Random, Uncertainty, Margin.


## Cell 11 – Linear Classifier (Self-Supervised Embedding Framework)

In [51]:
class LinearHead(nn.Module):
    def __init__(self, d=512, c=10):
        super().__init__()
        self.fc = nn.Linear(d, c)
    def forward(self, x):
        return self.fc(x)


def train_linear(features, labels, indices, epochs=200, lr=2.5, bs=64):
    """Train a linear classifier on labelled subset of SimCLR features."""
    X = torch.tensor(features[indices], dtype=torch.float32)
    Y = torch.tensor(labels[indices],   dtype=torch.long)
    loader = DataLoader(TensorDataset(X, Y),
                        batch_size=min(bs, len(indices)), shuffle=True)

    model = LinearHead(features.shape[1], 10).to(DEVICE)
    opt   = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model


def eval_linear(model, features, labels):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(features, dtype=torch.float32).to(DEVICE))
    return (preds.argmax(1).cpu().numpy() == labels).mean() * 100

print("Linear classifier ready.")


Linear classifier ready.


## Cell 12 – Fully-Supervised Framework (ResNet-18 from scratch)

In [52]:
def train_supervised(indices, trainset, testset,
                     epochs=200, lr=0.025, bs=64):
    """Train ResNet-18 from scratch on labelled subset → test accuracy."""
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616]),
    ])

    class Sub(Dataset):
        def __init__(self, ds, idxs, tf):
            self.ds, self.idxs, self.tf = ds, idxs, tf
        def __len__(self): return len(self.idxs)
        def __getitem__(self, i):
            img, lab = self.ds[self.idxs[i]]
            return self.tf(img), lab

    class Wrap(Dataset):
        def __init__(self, ds, tf):
            self.ds, self.tf = ds, tf
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            img, lab = self.ds[i]
            return self.tf(img), lab

    tr_loader = DataLoader(Sub(trainset, indices, train_tf),
                           batch_size=min(bs, len(indices)),
                           shuffle=True, num_workers=2)
    te_loader = DataLoader(Wrap(testset, test_tf),
                           batch_size=256, shuffle=False, num_workers=2)

    model = resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    model.maxpool = nn.Identity()
    model = model.to(DEVICE)

    opt   = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                      nesterov=True, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    model.train()
    for _ in range(epochs):
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in te_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / total * 100

print("Fully-supervised framework ready.")


Fully-supervised framework ready.


## Cell 13 – Active Learning Experiment Runner

In [53]:
def run_experiment(strategy, name, budget, rounds, reps,
                   framework='embedding'):
    """
    Run AL experiment over multiple rounds and repetitions.
    Returns dict with budgets, mean accuracies, std errors.
    """
    print(f"\n{'─'*55}")
    print(f"  {name} | B={budget} | rounds={rounds} | reps={reps} | {framework}")
    print(f"{'─'*55}")

    all_accs = []
    for r in range(reps):
        set_seed(r * 111 + 7)
        labeled = []
        accs = []
        clf = None  # current classifier (for uncertainty-based)

        for rd in range(rounds):
            # ── query ──
            if name in ('Uncertainty', 'Margin') and rd > 0:
                new = strategy.select(budget, labeled=labeled,
                                      model=clf, features=feat_train)
            else:
                new = strategy.select(budget, labeled=labeled if labeled else None)
            labeled.extend(new)

            # ── train & eval ──
            if framework == 'embedding':
                clf = train_linear(feat_train, lab_train, labeled)
                acc = eval_linear(clf, feat_test, lab_test)
            else:  # supervised
                acc = train_supervised(labeled, trainset_raw, testset_raw)

            accs.append(acc)
            if r == 0:
                print(f"    round {rd+1}: {len(labeled):>4} labels → {acc:.2f}%")

        all_accs.append(accs)

    arr = np.array(all_accs)
    return {
        'name':    name,
        'budgets': [(i+1)*budget for i in range(rounds)],
        'mean':    arr.mean(0).tolist(),
        'se':      (arr.std(0) / np.sqrt(reps)).tolist(),
        'raw':     arr.tolist(),
    }

print("Experiment runner ready.")


Experiment runner ready.


## Cell 14 – Experiment 1: Self-Supervised Embedding, B = M = 10

Reproducing Fig. 5a from the paper: 5 AL rounds, budget = 10 per round (= number of classes).


In [54]:
N = len(feat_train)
tc   = TypiClust(feat_train, lab_train, K=20)
rnd  = RandomAL(N)
unc  = UncertaintyAL(N)
mar  = MarginAL(N)

ROUNDS = 5
REPS   = 5

res_emb10 = {}
for name, strat in [('TPC_RP', tc), ('Random', rnd),
                     ('Uncertainty', unc), ('Margin', mar)]:
    res_emb10[name] = run_experiment(strat, name, budget=10,
                                      rounds=ROUNDS, reps=REPS,
                                      framework='embedding')
print("\n✓ Embedding B=10 experiments done.")


TypiClust ready – typicality range [1.118, 19.536]

───────────────────────────────────────────────────────
  TPC_RP | B=10 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   10 labels → 59.35%
    round 2:   20 labels → 70.44%
    round 3:   30 labels → 74.33%
    round 4:   40 labels → 78.85%
    round 5:   50 labels → 80.14%

───────────────────────────────────────────────────────
  Random | B=10 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   10 labels → 40.03%
    round 2:   20 labels → 47.22%
    round 3:   30 labels → 53.96%
    round 4:   40 labels → 65.52%
    round 5:   50 labels → 66.51%

───────────────────────────────────────────────────────
  Uncertainty | B=10 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   10 labels → 40.03%
    round 2:   20 labels → 46.67%
    round 3:   30 labels → 54.06%
    round 4:   40 l

## Cell 15 – Experiment 2: Self-Supervised Embedding, B = 5M = 50

In [55]:
res_emb50 = {}
for name, strat in [('TPC_RP', tc), ('Random', rnd),
                     ('Uncertainty', unc), ('Margin', mar)]:
    res_emb50[name] = run_experiment(strat, name, budget=50,
                                      rounds=ROUNDS, reps=REPS,
                                      framework='embedding')
print("\n✓ Embedding B=50 experiments done.")



───────────────────────────────────────────────────────
  TPC_RP | B=50 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   50 labels → 79.33%
    round 2:  100 labels → 80.87%
    round 3:  150 labels → 82.65%
    round 4:  200 labels → 83.17%
    round 5:  250 labels → 83.44%

───────────────────────────────────────────────────────
  Random | B=50 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   50 labels → 69.81%
    round 2:  100 labels → 77.24%
    round 3:  150 labels → 80.43%
    round 4:  200 labels → 81.64%
    round 5:  250 labels → 82.09%

───────────────────────────────────────────────────────
  Uncertainty | B=50 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   50 labels → 69.81%
    round 2:  100 labels → 73.28%
    round 3:  150 labels → 75.23%
    round 4:  200 labels → 79.32%
    round 5:  250 labels → 79.59%

─

## Cell 16 – Experiment 3: Fully Supervised, B = 10

ResNet-18 trained from scratch on labelled subset only.  
Fewer reps (3) because each repetition trains 5 fresh ResNets.


In [56]:
res_sup10 = {}
for name, strat in [('TPC_RP', tc), ('Random', rnd)]:
    res_sup10[name] = run_experiment(strat, name, budget=10,
                                     rounds=ROUNDS, reps=3,
                                     framework='supervised')
print("\n✓ Supervised B=10 experiments done.")



───────────────────────────────────────────────────────
  TPC_RP | B=10 | rounds=5 | reps=3 | supervised
───────────────────────────────────────────────────────
    round 1:   10 labels → 15.44%
    round 2:   20 labels → 21.01%
    round 3:   30 labels → 22.20%
    round 4:   40 labels → 25.36%
    round 5:   50 labels → 25.00%

───────────────────────────────────────────────────────
  Random | B=10 | rounds=5 | reps=3 | supervised
───────────────────────────────────────────────────────
    round 1:   10 labels → 13.49%
    round 2:   20 labels → 17.47%
    round 3:   30 labels → 20.25%
    round 4:   40 labels → 19.31%
    round 5:   50 labels → 25.78%

✓ Supervised B=10 experiments done.


## Cell 17 – Plotting Helper

In [57]:
COLORS  = {'TPC_RP':'#1f77b4', 'Random':'#ff7f0e',
            'Uncertainty':'#2ca02c', 'Margin':'#d62728',
            'TPC_RP_Mod':'#9467bd'}
MARKERS = {'TPC_RP':'o', 'Random':'s', 'Uncertainty':'^',
           'Margin':'v', 'TPC_RP_Mod':'D'}

def plot_al(results, title, path):
    plt.figure(figsize=(7, 4.5))
    for r in results.values():
        n = r['name']
        plt.plot(r['budgets'], r['mean'],
                 marker=MARKERS.get(n,'o'), color=COLORS.get(n,'gray'),
                 lw=2, ms=6, label=n)
        lo = np.array(r['mean']) - np.array(r['se'])
        hi = np.array(r['mean']) + np.array(r['se'])
        plt.fill_between(r['budgets'], lo, hi,
                         color=COLORS.get(n,'gray'), alpha=0.15)
    plt.xlabel('Cumulative Budget', fontsize=11)
    plt.ylabel('Accuracy (%)', fontsize=11)
    plt.title(title, fontsize=12)
    plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {path}")

print("Plotting helper ready.")

Plotting helper ready.


## Cell 18 – Plots: Embedding Framework

In [58]:
plot_al(res_emb10,
        'CIFAR-10 - Self-Supervised Embedding (B=10)',
        os.path.join(SAVE_DIR, 'plot_emb_b10.png'))

plot_al(res_emb50,
        'CIFAR-10 - Self-Supervised Embedding (B=50)',
        os.path.join(SAVE_DIR, 'plot_emb_b50.png'))

Saved: /kaggle/working/plot_emb_b10.png
Saved: /kaggle/working/plot_emb_b50.png


## Cell 19 – Plots: Fully Supervised Framework

In [59]:
plot_al(res_sup10,
        'CIFAR-10 - Fully Supervised (B=10)',
        os.path.join(SAVE_DIR, 'plot_sup_b10.png'))

Saved: /kaggle/working/plot_sup_b10.png


## Cell 20 – Class Distribution: TypiClust vs Random

In [60]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ground truth
u, c = np.unique(lab_train, return_counts=True)
axes[0].bar(u, c/c.sum(), color='steelblue', alpha=0.7)
axes[0].set_title('Ground Truth'); axes[0].set_xlabel('Class')
axes[0].set_ylabel('Proportion')

# TypiClust 30
set_seed(42)
sel_tc = tc.select(30)
u2, c2 = np.unique(lab_train[sel_tc], return_counts=True)
full_c = np.zeros(10)
for a, b in zip(u2, c2): full_c[a] = b
axes[1].bar(range(10), full_c/full_c.sum(), color='#1f77b4', alpha=0.7)
axes[1].set_title('TypiClust (30 samples)'); axes[1].set_xlabel('Class')

# Random 30
set_seed(42)
sel_rn = rnd.select(30)
u3, c3 = np.unique(lab_train[sel_rn], return_counts=True)
full_c2 = np.zeros(10)
for a, b in zip(u3, c3): full_c2[a] = b
axes[2].bar(range(10), full_c2/full_c2.sum(), color='#ff7f0e', alpha=0.7)
axes[2].set_title('Random (30 samples)'); axes[2].set_xlabel('Class')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_class_dist.png'), dpi=300)
plt.show()
print("Saved: plot_class_dist.png")

Saved: plot_class_dist.png


## Cell 21 – Total Variation Distance (Fig. 7b reproduction)

In [61]:
def tv_distance(selected_labels, n_classes=10):
    """Total Variation distance between selected class dist and uniform."""
    counts = np.bincount(selected_labels, minlength=n_classes).astype(float)
    p = counts / counts.sum()
    q = np.ones(n_classes) / n_classes
    return 0.5 * np.abs(p - q).sum()

budgets_tv = [10, 20, 30, 40, 50]
tv_tc_list, tv_rn_list = [], []

for b in budgets_tv:
    tvs_tc, tvs_rn = [], []
    for seed in range(10):
        set_seed(seed)
        sel = tc.select(b)
        tvs_tc.append(tv_distance(lab_train[sel]))
        set_seed(seed)
        sel = rnd.select(b)
        tvs_rn.append(tv_distance(lab_train[sel]))
    tv_tc_list.append(np.mean(tvs_tc))
    tv_rn_list.append(np.mean(tvs_rn))

plt.figure(figsize=(6, 4))
plt.plot(budgets_tv, tv_tc_list, 'o-', color=COLORS['TPC_RP'],  label='TPC_RP', lw=2)
plt.plot(budgets_tv, tv_rn_list, 's-', color=COLORS['Random'],  label='Random', lw=2)
plt.xlabel('Budget'); plt.ylabel('TV Distance')
plt.title('Total Variation Distance to Uniform Class Distribution')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_tv_distance.png'), dpi=300)
plt.show()
print("Saved: plot_tv_distance.png")

Saved: plot_tv_distance.png


## Cell 22 – Results Summary Table

In [62]:
print("=" * 65)
print(f"{'Strategy':<16} {'Emb B=10':>12} {'Emb B=50':>12} {'Sup B=10':>12}")
print("=" * 65)

for name in ['TPC_RP', 'Random', 'Uncertainty', 'Margin']:
    e10 = res_emb10[name]['mean'][-1]
    e50 = res_emb50[name]['mean'][-1]
    s10 = res_sup10.get(name, {}).get('mean', [None])[-1]
    s10_str = f"{s10:.2f}%" if s10 is not None else "—"
    print(f"{name:<16} {e10:>11.2f}% {e50:>11.2f}% {s10_str:>12}")

print("=" * 65)


Strategy             Emb B=10     Emb B=50     Sup B=10
TPC_RP                 80.21%       83.44%       25.61%
Random                 70.84%       81.38%       22.61%
Uncertainty            65.13%       79.59%            —
Margin                 72.57%       82.69%            —


## Cell 23 – Statistical Analysis

In [63]:
print("Paired t-tests: TPC_RP vs each baseline (final round, Emb B=10)")
print("─" * 55)

tc_raw = np.array(res_emb10['TPC_RP']['raw'])[:, -1]   # final round
for name in ['Random', 'Uncertainty', 'Margin']:
    other_raw = np.array(res_emb10[name]['raw'])[:, -1]
    t_stat, p_val = stats.ttest_rel(tc_raw, other_raw)
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01
           else ("*" if p_val < 0.05 else "n.s."))
    diff = tc_raw.mean() - other_raw.mean()
    print(f"TPC_RP vs {name:<14} Δ={diff:+.2f}%  t={t_stat:.3f}  "
          f"p={p_val:.4f}  {sig}")

print("\nEffect sizes (Cohen's d):")
for name in ['Random', 'Uncertainty', 'Margin']:
    other_raw = np.array(res_emb10[name]['raw'])[:, -1]
    diff = tc_raw - other_raw
    d = diff.mean() / (diff.std() + 1e-10)
    print(f"  TPC_RP vs {name:<14} d = {d:.3f}")


Paired t-tests: TPC_RP vs each baseline (final round, Emb B=10)
───────────────────────────────────────────────────────
TPC_RP vs Random         Δ=+9.37%  t=6.937  p=0.0023  **
TPC_RP vs Uncertainty    Δ=+15.08%  t=3.877  p=0.0179  *
TPC_RP vs Margin         Δ=+7.65%  t=6.000  p=0.0039  **

Effect sizes (Cohen's d):
  TPC_RP vs Random         d = 3.468
  TPC_RP vs Uncertainty    d = 1.939
  TPC_RP vs Margin         d = 3.000


## Cell 24 – Algorithm Modification: Density-Weighted Adaptive TypiClust

**Three improvements over original TPC_RP:**

1. **Gaussian-weighted typicality** – closer neighbours contribute more, reducing noise from distant neighbours.
2. **Adaptive K** – `K = max(5, min(20, cluster_size // 5))` avoids unreliable density estimates in small clusters.
3. **Cluster-density prioritisation** – when choosing *which* clusters to sample from, prefer denser clusters as they represent more "typical" regions of the data manifold.

**Rationale:** The original algorithm uses fixed K=20 for all clusters and treats all clusters equally. Small clusters may have <20 meaningful neighbours, producing noisy typicality scores. Additionally, equally weighting all neighbours ignores the fact that very close neighbours are much stronger evidence of density than distant ones.


In [64]:
class TypiClustMod(TypiClust):
    """
    Modified TPC_RP with:
      - Gaussian-weighted typicality
      - Adaptive K per cluster
      - Cluster-density scoring for priority
    """
    def __init__(self, features, labels, K=20, max_clusters=500,
                 sigma=1.0, density_w=0.3):
        # skip parent __init__, compute our own typicality
        self.features     = features
        self.labels       = labels
        self.N            = len(features)
        self.K            = K
        self.max_clusters = max_clusters
        self.sigma        = sigma
        self.density_w    = density_w

        k = min(K + 1, self.N)
        nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
        nn.fit(features)
        self.nn_dists, self.nn_idxs = nn.kneighbors(features)

        # Gaussian-weighted average distance
        d = self.nn_dists[:, 1:]
        w = np.exp(-d**2 / (2 * sigma**2))
        wavg = (d * w).sum(1) / (w.sum(1) + 1e-10)
        self.typicality = 1.0 / (wavg + 1e-10)
        print(f"ModTypiClust ready – typicality range "
              f"[{self.typicality.min():.3f}, {self.typicality.max():.3f}]")

    def select(self, budget, labeled=None):
        labeled_set = set(labeled) if labeled else set()
        n_clust = min((len(labeled_set) + budget) if labeled else budget,
                      self.max_clusters)

        km = (KMeans(n_clust, n_init=10, random_state=42) if n_clust <= 50
              else MiniBatchKMeans(n_clust, batch_size=1024, n_init=3, random_state=42))
        cl = km.fit_predict(self.features)

        # cluster densities
        c_dens = {}
        for cid in range(n_clust):
            mask = (cl == cid)
            c_dens[cid] = self.typicality[mask].mean() if mask.sum() >= 5 else 0.0

        candidates = []
        for cid in range(n_clust):
            idxs = np.where(cl == cid)[0]
            if len(idxs) < 5:
                continue
            if labeled_set and any(i in labeled_set for i in idxs):
                continue

            # adaptive K
            ak = max(5, min(self.K, len(idxs) // 5))
            if ak < self.K:
                d_loc = self.nn_dists[idxs, 1:ak+1]
                w_loc = np.exp(-d_loc**2 / (2*self.sigma**2))
                wavg_loc = (d_loc * w_loc).sum(1) / (w_loc.sum(1) + 1e-10)
                typ_loc = 1.0 / (wavg_loc + 1e-10)
            else:
                typ_loc = self.typicality[idxs]

            best_local = np.argmax(typ_loc)
            best_global = idxs[best_local]
            if best_global in labeled_set:
                continue

            score = typ_loc[best_local] + self.density_w * c_dens[cid]
            candidates.append((best_global, score))

        candidates.sort(key=lambda x: x[1], reverse=True)
        picked = [c[0] for c in candidates[:budget]]

        if len(picked) < budget:
            pool = list(set(range(self.N)) - labeled_set - set(picked))
            order = np.argsort(self.typicality[pool])[::-1]
            for i in order:
                if len(picked) >= budget: break
                picked.append(pool[i])

        return picked[:budget]


tc_mod = TypiClustMod(feat_train, lab_train, K=20, sigma=1.0, density_w=0.3)
print("Modified TypiClust initialised.")


ModTypiClust ready – typicality range [1.120, 19.539]
Modified TypiClust initialised.


## Cell 25 – Run Modified TypiClust Experiments

In [65]:
res_mod10 = run_experiment(tc_mod, 'TPC_RP_Mod', budget=10,
                           rounds=ROUNDS, reps=REPS, framework='embedding')
res_mod50 = run_experiment(tc_mod, 'TPC_RP_Mod', budget=50,
                           rounds=ROUNDS, reps=REPS, framework='embedding')

print("\n✓ Modified experiments done.")



───────────────────────────────────────────────────────
  TPC_RP_Mod | B=10 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   10 labels → 59.35%
    round 2:   20 labels → 70.44%
    round 3:   30 labels → 73.26%
    round 4:   40 labels → 78.89%
    round 5:   50 labels → 79.51%

───────────────────────────────────────────────────────
  TPC_RP_Mod | B=50 | rounds=5 | reps=5 | embedding
───────────────────────────────────────────────────────
    round 1:   50 labels → 79.36%
    round 2:  100 labels → 81.38%
    round 3:  150 labels → 82.65%
    round 4:  200 labels → 83.25%
    round 5:  250 labels → 83.03%

✓ Modified experiments done.


## Cell 26 – Original vs Modified Comparison

In [66]:
# B=10 comparison
cmp10 = {'TPC_RP': res_emb10['TPC_RP'],
         'TPC_RP_Mod': res_mod10,
         'Random': res_emb10['Random']}
plot_al(cmp10, 'Original vs Modified TPC_RP (B=10)',
        os.path.join(SAVE_DIR, 'plot_mod_vs_orig_b10.png'))

# B=50 comparison
cmp50 = {'TPC_RP': res_emb50['TPC_RP'],
         'TPC_RP_Mod': res_mod50,
         'Random': res_emb50['Random']}
plot_al(cmp50, 'Original vs Modified TPC_RP (B=50)',
        os.path.join(SAVE_DIR, 'plot_mod_vs_orig_b50.png'))

Saved: /kaggle/working/plot_mod_vs_orig_b10.png
Saved: /kaggle/working/plot_mod_vs_orig_b50.png


## Cell 27 – Statistical Test: Original vs Modified

In [67]:
print("Paired t-test: TPC_RP vs TPC_RP_Mod (Emb B=10, final round)")
print("─" * 50)

orig_raw = np.array(res_emb10['TPC_RP']['raw'])[:, -1]
mod_raw  = np.array(res_mod10['raw'])[:, -1]

t_stat, p_val = stats.ttest_rel(mod_raw, orig_raw)
diff = mod_raw.mean() - orig_raw.mean()
d = diff / (np.std(mod_raw - orig_raw) + 1e-10)

print(f"Original mean: {orig_raw.mean():.2f}%")
print(f"Modified mean: {mod_raw.mean():.2f}%")
print(f"Difference:    {diff:+.2f}%")
print(f"t-statistic:   {t_stat:.3f}")
print(f"p-value:       {p_val:.4f}")
print(f"Cohen's d:     {d:.3f}")
print(f"Significant:   {'Yes' if p_val < 0.05 else 'No'} (α=0.05)")


Paired t-test: TPC_RP vs TPC_RP_Mod (Emb B=10, final round)
──────────────────────────────────────────────────
Original mean: 80.21%
Modified mean: 79.55%
Difference:    -0.67%
t-statistic:   -50.202
p-value:       0.0000
Cohen's d:     -25.101
Significant:   Yes (α=0.05)


## Cell 28 – Full Results Table

In [68]:
print("\n" + "=" * 70)
print("FULL RESULTS TABLE – CIFAR-10")
print("=" * 70)
print(f"{'Strategy':<18} {'Emb B=10':>12} {'Emb B=50':>12} {'Sup B=10':>12}")
print("-" * 70)

for name in ['TPC_RP', 'Random', 'Uncertainty', 'Margin']:
    e10 = res_emb10[name]['mean'][-1]
    e50 = res_emb50[name]['mean'][-1]
    s10 = res_sup10.get(name, {}).get('mean', [None])[-1]
    e10_se = res_emb10[name]['se'][-1]
    e50_se = res_emb50[name]['se'][-1]
    s10_str = f"{s10:.1f}" if s10 else "—"
    print(f"{name:<18} {e10:.1f}±{e10_se:.1f}%  {e50:.1f}±{e50_se:.1f}%  {s10_str:>8}")

# Modification
print("-" * 70)
e10m = res_mod10['mean'][-1]
e50m = res_mod50['mean'][-1]
e10m_se = res_mod10['se'][-1]
e50m_se = res_mod50['se'][-1]
print(f"{'TPC_RP_Mod':<18} {e10m:.1f}±{e10m_se:.1f}%  {e50m:.1f}±{e50m_se:.1f}%")
print("=" * 70)



FULL RESULTS TABLE – CIFAR-10
Strategy               Emb B=10     Emb B=50     Sup B=10
----------------------------------------------------------------------
TPC_RP             80.2±0.0%  83.4±0.0%      25.6
Random             70.8±1.2%  81.4±0.4%      22.6
Uncertainty        65.1±3.5%  79.6±0.1%         —
Margin             72.6±1.1%  82.7±0.1%         —
----------------------------------------------------------------------
TPC_RP_Mod         79.5±0.0%  83.0±0.0%


## Cell 29 – Save All Results

In [69]:
results_all = {
    'emb_b10':   res_emb10,
    'emb_b50':   res_emb50,
    'sup_b10':   res_sup10,
    'mod_b10':   res_mod10,
    'mod_b50':   res_mod50,
}

with open(os.path.join(SAVE_DIR, 'all_results.json'), 'w') as f:
    json.dump(results_all, f, indent=2, default=str)

# List all saved files
print("\nAll files in /kaggle/working/:")
for f in sorted(os.listdir(SAVE_DIR)):
    if not f.startswith('.') and f != 'data':
        size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024 / 1024
        print(f"  {size:>6.1f} MB  {f}")

print("\nCOURSEWORK COMPLETE!")


All files in /kaggle/working/:
     0.0 MB  all_results.json
    19.5 MB  feat_test.npy
    97.7 MB  feat_train.npy
     0.1 MB  lab_test.npy
     0.4 MB  lab_train.npy
     0.1 MB  plot_class_dist.png
     0.3 MB  plot_emb_b10.png
     0.3 MB  plot_emb_b50.png
     0.2 MB  plot_mod_vs_orig_b10.png
     0.2 MB  plot_mod_vs_orig_b50.png
     0.1 MB  plot_simclr_loss.png
     0.2 MB  plot_sup_b10.png
     0.1 MB  plot_tv_distance.png
    87.8 MB  simclr_latest.pt

COURSEWORK COMPLETE!
